# HachimiMT — Benchmark tốc độ CPU (Colab)

Đo `chunk/giây` trên **CPU** ở nhiều mức `threads` × `batch` để xem:
1. Colab CPU runtime mạnh hơn HF Space (2 vCPU) bao nhiêu — in số vCPU thực.
2. Trên CPU, tăng `threads`/`batch` có tăng tốc không (để chỉnh `hardware.py` nếu đáng).

**Cách dùng:** KHÔNG cần GPU. `Runtime → Change runtime type → CPU` (hoặc để mặc định), rồi `Runtime → Run all`. Cell cuối in bảng — **báo lại bảng đó** để chốt cấu hình CPU.

> Ghi chú: notebook này **ép chạy CPU** (mask CUDA) nên dù đang ở GPU runtime vẫn đo đúng CPU. Muốn đo GPU thì dùng `HachimiMT_Benchmark_T4.ipynb`.

In [ ]:
# 1. Ép CPU (mask CUDA TRƯỚC khi import ct2) + tải code + cài thư viện
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"   # ép CT2 chạy CPU dù runtime có GPU
import urllib.request, zipfile, shutil
ZIP_URL = "https://huggingface.co/spaces/ngocdang83/HachimiMT-demo/resolve/main/hachimimt-local.zip"
urllib.request.urlretrieve(ZIP_URL, "hachimimt-local.zip")
shutil.rmtree("hachimimt", ignore_errors=True)
with zipfile.ZipFile("hachimimt-local.zip") as z: z.extractall(".")
!pip install -q -r hachimimt/requirements.txt
print("OK — đã tải code + cài thư viện.")

In [ ]:
# 2. In số CPU thực (quota container) — để biết Colab cấp bao nhiêu vCPU
import os, math
from pathlib import Path

def effective_cpu_count():
    vals = []
    if hasattr(os, "sched_getaffinity"):
        try: vals.append(len(os.sched_getaffinity(0)))
        except OSError: pass
    try:
        quota, period = Path("/sys/fs/cgroup/cpu.max").read_text().split()
        if quota != "max":
            vals.append(max(1, math.ceil(int(quota) / int(period))))
    except (OSError, ValueError): pass
    vals.append(os.cpu_count() or 1)
    return max(1, min(vals)), vals

eff, raw = effective_cpu_count()
print(f"os.cpu_count()         = {os.cpu_count()}")
print(f"sched_getaffinity      = {len(os.sched_getaffinity(0)) if hasattr(os,'sched_getaffinity') else 'n/a'}")
print(f"=> CPU hiệu dụng (min) = {eff}   (các nguồn: {raw})")
print(f"\nHF Space cpu-basic = 2 vCPU. Colab CPU runtime ở đây = {eff} vCPU"
      f" ({'MẠNH HƠN' if eff > 2 else 'ngang' if eff == 2 else 'yếu hơn'} Space).")
CPU_EFF = eff

In [ ]:
# 3. Dựng văn bản test đa dạng (nhiều câu khác nhau, không lặp 1 câu) + tải model 1 lần
import sys
sys.path.insert(0, os.path.abspath("hachimimt/src"))

PARA = (
    "他抬头看向远处的山门，心中升起一股莫名的悸动。"
    "师兄说得对，我等修士当以大道为重，岂能因私情误了修行。"
    "她转身离去，没有再回头看一眼，仿佛一切已成定局。"
    "凌伊山掏出手机，查询起了最近开往雪霏市的机票。"
    "玄中门欲炼的丹药乃是凝婴丹，主材之一便是天婴果。"
    "他必须得抓紧时间了，否则一切都将来不及挽回。"
    "姑娘家世显赫，乃是百年难遇的修炼奇才，前途不可限量。"
    "公子莫要心急，此事还需从长计议，切不可操之过急。"
)
# CPU chậm hơn GPU ~50-70× → dùng đoạn NHỎ để mỗi lần đo chỉ ~vài-chục giây.
TEXT = PARA * 30          # ~240 câu
from translator import HachimiTranslator, Backend, DEFAULT_MODEL_KEY
from hardware import detect_hardware_profile, recommend_batch_size, recommend_ct2_threads

# tải model 1 lần để cache (lần load trong vòng benchmark sẽ nhanh)
_tmp = HachimiTranslator(detect_hardware_profile())
_tmp.load(DEFAULT_MODEL_KEY, Backend.CT2)
N = _tmp.count_chunks(TEXT, "sentence")
N_CHARS = len(TEXT)
print(f"Văn bản test: {N_CHARS:,} chữ Hán · {N} chunk câu")
print(f"Mặc định app (CPU {CPU_EFF} luồng): "
      f"threads={recommend_ct2_threads(CPU_EFF, has_cuda=False)}, "
      f"batch={recommend_batch_size(CPU_EFF, has_cuda=False, vram_gb=None)}")
del _tmp

In [ ]:
# 4. Benchmark: vòng NGOÀI = threads (phải reload model), vòng TRONG = batch (đổi runtime)
#    CT2 nhận intra_threads lúc khởi tạo → mỗi mức threads tạo translator mới.
import time, gc

def make_translator(threads):
    os.environ["HACHIMIMT_THREADS"] = str(threads)
    os.environ["HACHIMIMT_INTER_THREADS"] = "1"   # CT2: nhiều inter trên 1 CPU ít lợi
    prof = detect_hardware_profile()              # đọc lại env threads
    tr = HachimiTranslator(prof)
    tr.load(DEFAULT_MODEL_KEY, Backend.CT2)
    tr._ct2_batch_type = "examples"               # CPU: examples ổn định
    return tr

def measure(tr, batch):
    tr._batch_size = batch
    tr.translate_text(TEXT[:120], chunk_mode="sentence", beam_size=1)   # warmup
    t0 = time.perf_counter()
    rows, _ = tr.translate_text(TEXT, chunk_mode="sentence", beam_size=1)
    dt = time.perf_counter() - t0
    return len(rows) / dt, N_CHARS / dt, dt

# threads quét: 1, 2, và tới số vCPU thực (thêm vài mốc nếu CPU nhiều)
thread_set = sorted({1, 2, CPU_EFF, max(2, CPU_EFF // 2)} | ({4, 8} if CPU_EFF >= 4 else set()))
thread_set = [t for t in thread_set if t <= max(CPU_EFF, 2)]
batch_set = [4, 8, 16, 32, 64]

print(f"{'threads':>7} {'batch':>6} {'chunk/s':>9} {'chữ/s':>9} {'giây':>7}")
print("-" * 44)
results = []
best = None
for th in thread_set:
    tr = make_translator(th)
    for b in batch_set:
        try:
            cps, chars, dt = measure(tr, b)
            print(f"{th:>7} {b:>6} {cps:>9.1f} {chars:>9.0f} {dt:>7.1f}")
            results.append((th, b, cps, chars))
            if best is None or cps > best[2]:
                best = (th, b, cps, chars)
        except Exception as e:
            print(f"{th:>7} {b:>6}  LỖI: {str(e)[:34]}")
    del tr; gc.collect()

print("-" * 44)
if best:
    print(f"NHANH NHẤT: threads={best[0]} batch={best[1]} → {best[2]:.1f} chunk/s ({best[3]:.0f} chữ/s)")
    print(f"1 triệu chữ ~ {1_000_000/best[3]/60:.1f} phút · bộ truyện 2.4M chữ ~ {2_400_000/best[3]/60:.0f} phút")
print("\n→ Báo lại bảng này: (a) số vCPU thực ở cell 2, (b) cấu hình threads/batch nhanh nhất.")